# Main-story length over years

HowLongToBeat `main_story` hours by release year. Cleaning rules live in `hltb` — do not re-filter here.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

# editable install preferred; fallback for ad-hoc kernels:
repo = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo / "src"))

from hltb import clean_games, load_games, yearly_main_story, yearly_main_story_all

raw = load_games()
cleaned = clean_games(raw)
print(f"raw rows: {len(raw):,}")
print(f"cleaned rows: {len(cleaned):,}")
print(f"main_story coverage in raw: {raw['main_story'].notna().mean():.1%}")
print(cleaned[["main_story", "release_year"]].describe())

In [ ]:
ax = cleaned["main_story"].clip(upper=cleaned["main_story"].quantile(0.99)).hist(bins=40)
ax.set_xlabel("main_story hours (clipped at p99 for display)")
ax.set_ylabel("games")
ax.set_title("Distribution of main_story")
plt.show()

In [ ]:
trend = yearly_main_story(cleaned, min_n=30)
all_years = yearly_main_story_all(cleaned)
thin = all_years.loc[all_years["n"] < 30]

if trend.empty:
    print("No years with at least 30 games after cleaning — nothing to chart.")
else:
    fig, ax1 = plt.subplots(figsize=(10, 4))
    ax1.plot(trend["year"], trend["median_main_story"], marker="o")
    ax1.set_xlabel("release year")
    ax1.set_ylabel("median main_story (hours)")
    ax1.set_title("Median main_story over years (n ≥ 30)")

    ax2 = ax1.twinx()
    ax2.bar(trend["year"], trend["n"], alpha=0.2)
    ax2.set_ylabel("games (n)")
    plt.show()

print("Thin years excluded from main trend:")
display(thin)

In [ ]:
decade = cleaned.copy()
decade["decade"] = (decade["release_year"] // 10) * 10
decade_med = decade.groupby("decade")["main_story"].median()
print("Decade medians:")
print(decade_med)

print("Longest median years:")
display(trend.nlargest(5, "median_main_story"))
print("Shortest median years:")
display(trend.nsmallest(5, "median_main_story"))

## Findings

1. **Headline:** The long-run increase is clearest in decade medians: about 1.0h in the 1980s, 3.0h in the 1990s, and 7.0h in the 2000s; the 2010s median is 4.5h.
2. **Annual endpoints:** The included series runs from 0.5h in 1982 to 6.5h in 2019, but these are not like-for-like endpoints. Early years are dominated by very short arcade-era titles; 2019 has 174 games versus 736 in 2018, whose median is 4.0h.
3. **Sample:** Cleaning keeps 15,681 games; raw `main_story` coverage is ~48.2%.
4. **Caveats:** Missing durations are common; release year prefers NA→EU→JP; years with n<30 are excluded; and late years—especially 2019—may be thin or incomplete in this 2020-dated dump.